In [ ]:
# model 2 with transfer learning

In [ ]:
SUPER_THRESHOLD_logit = 2.5

SUB_THRESHOLD_prob = 0.5

SUPER_THRESHOLD_softmax= 0.9

NUM_EPOCHS=30000

Cell 1: Imports & Dataset Class

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# 1. Configuration
# We set the seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
print(f"Using device: {device}")

# 2. Define the Dataset Class
class MultiClassImageDataset(Dataset):
    def __init__(self, ann_df, super_map_df, sub_map_df, img_dir, transform=None):
        self.ann_df = ann_df 
        self.super_map_df = super_map_df
        self.sub_map_df = sub_map_df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ann_df)

    def __getitem__(self, idx):
        img_name = self.ann_df['image'][idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Open image and ensure RGB
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_name}: {e}")
            # Return a black image in case of error to prevent crash
            image = Image.new('RGB', (64, 64))
        
        # Get labels
        super_idx = self.ann_df['superclass_index'][idx]
        sub_idx = self.ann_df['subclass_index'][idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, super_idx, sub_idx

Cell2: Model Architecture (Soft-Gated MoE)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

class SoftGatedExpertModel(nn.Module):
    def __init__(self, num_super_classes, sub_map_df, hidden_dim=256):
        super().__init__()
        
        # 1. Backbone: ResNet18 (Pre-trained)
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.feature_dim = backbone.fc.in_features
        
        # 2. Gating Network
        self.gate = nn.Sequential(
            nn.Linear(self.feature_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, num_super_classes) 
        )
        
        # 3. Build Experts
        self.experts = nn.ModuleList()
        self.expert_indices = []
        
        unique_supers = sorted(sub_map_df['superclass_index'].unique())
        self.total_sub_classes = len(sub_map_df) 
        
        for super_idx in unique_supers:
            indices = sub_map_df[sub_map_df['superclass_index'] == super_idx].index.tolist()
            
            # --- FIX: Explicitly set dtype to torch.long ---
            self.expert_indices.append(torch.tensor(indices, dtype=torch.long))
            # ------------------------------------------------
            
            expert_head = nn.Sequential(
                nn.Linear(self.feature_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Linear(hidden_dim, len(indices)) 
            )
            self.experts.append(expert_head)

    def forward(self, x):
        # 1. Extract Features
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        
        # 2. Gating
        gate_logits = self.gate(features) 
        gate_probs = F.softmax(gate_logits, dim=1)
        
        # 3. Expert Execution & Soft Aggregation
        final_sub_probs = torch.zeros(x.size(0), self.total_sub_classes, device=x.device)
        
        for i, expert in enumerate(self.experts):
            # Use safe indexing for gate probs (assuming sorted unique_supers aligns with 0,1,2...)
            gate_weight = gate_probs[:, i].unsqueeze(1)
            
            expert_logits = expert(features)
            expert_probs = F.softmax(expert_logits, dim=1)
            
            weighted_expert_probs = expert_probs * gate_weight
            
            indices = self.expert_indices[i].to(x.device)
            
            # This line crashed before; now it should work because indices are Long
            final_sub_probs[:, indices] += weighted_expert_probs
            
        return gate_logits, final_sub_probs

cell3: Data Loading

In [ ]:
# 1. Load Dataframes
# Ensure these files are in your working directory
train_ann_df = pd.read_csv('train_data.csv')
super_map_df = pd.read_csv('superclass_mapping.csv')
sub_map_df = pd.read_csv('subclass_mapping.csv')

# 2. Setup Paths
train_img_dir = 'train_images'

# 3. Define Transforms
# We use standard ResNet normalization
image_preprocessing = transforms.Compose([
    transforms.Resize((64, 64)), # Ensure size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 4. Create Datasets
full_dataset = MultiClassImageDataset(
    train_ann_df, 
    super_map_df, 
    sub_map_df, 
    train_img_dir, 
    transform=image_preprocessing
)

# 5. Split Train/Val
# We use a 90/10 split
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 6. Create DataLoaders (THIS WAS MISSING)
batch_size = 64
num_workers = 0 # Set to 0 if you are on Windows and get errors

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers
)

print(f"Data loaded. Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")
print(f"Number of Super Classes: {len(super_map_df)}")
print(f"Number of Sub Classes: {len(sub_map_df)}")

cell4: Trainer & Loss Setup

In [ ]:
# --- FIX: Infer the Hierarchy from Training Data ---
# The sub_map_df is missing 'superclass_index', so we find it from train_ann_df
# We get unique pairs of (subclass_index, superclass_index)
if 'superclass_index' not in sub_map_df.columns:
    print("Inferring subclass hierarchy from training data...")
    hierarchy_map = train_ann_df[['subclass_index', 'superclass_index']].drop_duplicates()
    
    # We ensure it's sorted by subclass_index so the indices match the dataframe rows
    hierarchy_map = hierarchy_map.sort_values('subclass_index').set_index('subclass_index')
    
    # We add this column to sub_map_df
    sub_map_df['superclass_index'] = hierarchy_map['superclass_index']
    print("Hierarchy added to sub_map_df.")
else:
    print("Hierarchy already present in sub_map_df.")

# Sanity Check
print(f"Sub-class 0 belongs to Super-class: {sub_map_df.loc[0, 'superclass_index']}")
# ---------------------------------------------------

# 1. Instantiate the Model
# We need the number of super classes and the sub-class mapping
num_super_classes = len(super_map_df)
model = SoftGatedExpertModel(num_super_classes, sub_map_df).to(device)

# 2. Define Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4) # Lower LR for fine-tuning

# 3. Define Loss Functions
criterion_super = nn.CrossEntropyLoss()
criterion_sub = nn.NLLLoss() # Expects log-probabilities

# 4. Trainer Class
class Trainer:
    def __init__(self, model, optimizer, train_loader, val_loader, device):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
    def train_epoch(self):
        self.model.train() # Important: Set to train mode
        running_loss = 0.0
        
        for images, super_labels, sub_labels in self.train_loader:
            images = images.to(self.device)
            super_labels = super_labels.to(self.device)
            sub_labels = sub_labels.to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward Pass
            gate_logits, sub_probs = self.model(images)
            
            # Calculate Loss
            # 1. Super-Class Loss (CrossEntropy on logits)
            loss_super = criterion_super(gate_logits, super_labels)
            
            # 2. Sub-Class Loss (NLL on log-probs)
            # Add epsilon to avoid log(0) error
            sub_log_probs = torch.log(sub_probs + 1e-9)
            loss_sub = criterion_sub(sub_log_probs, sub_labels)
            
            # Total Loss (You can weigh these if you want, e.g., 0.5 * super + 0.5 * sub)
            total_loss = loss_super + loss_sub
            
            total_loss.backward()
            self.optimizer.step()
            
            running_loss += total_loss.item()
            
        return running_loss / len(self.train_loader)

    def validate_epoch(self):
        self.model.eval() # Important: Set to eval mode (Fixes Batch Norm bug)
        
        running_loss = 0.0
        super_correct = 0
        sub_correct = 0
        total = 0
        
        with torch.no_grad():
            for images, super_labels, sub_labels in self.val_loader:
                images = images.to(self.device)
                super_labels = super_labels.to(self.device)
                sub_labels = sub_labels.to(self.device)
                
                # Forward
                gate_logits, sub_probs = self.model(images)
                
                # Loss
                loss_super = criterion_super(gate_logits, super_labels)
                sub_log_probs = torch.log(sub_probs + 1e-9)
                loss_sub = criterion_sub(sub_log_probs, sub_labels)
                total_loss = loss_super + loss_sub
                running_loss += total_loss.item()
                
                # Calculate Accuracy
                # Super-class prediction: argmax of gate logits
                _, super_pred = torch.max(gate_logits, 1)
                
                # Sub-class prediction: argmax of weighted probs
                _, sub_pred = torch.max(sub_probs, 1)
                
                total += super_labels.size(0)
                super_correct += (super_pred == super_labels).sum().item()
                sub_correct += (sub_pred == sub_labels).sum().item()
                
        metrics = {
            'val_loss': running_loss / len(self.val_loader),
            'super_acc': 100 * super_correct / total,
            'sub_acc': 100 * sub_correct / total
        }
        return metrics

trainer = Trainer(model, optimizer, train_loader, val_loader, device)
print("Trainer initialized.")

cell5: Training Loop

In [ ]:
# 5. Main Training Loop
num_epochs = NUM_EPOCHS # You can increase this (e.g., to 30) if you have time
best_val_acc = 0.0

print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    # 1. Run Training Step
    train_loss = trainer.train_epoch()
    
    # 2. Run Validation Step
    val_metrics = trainer.validate_epoch()
    
    # 3. Print Results
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_metrics['val_loss']:.4f}")
    print(f"  Val Super Acc:  {val_metrics['super_acc']:.2f}%")
    print(f"  Val Sub Acc:    {val_metrics['sub_acc']:.2f}%")
    
    # 4. Save Best Model
    # We prioritize Sub-Class Accuracy as it's the main goal
    if val_metrics['sub_acc'] > best_val_acc:
        best_val_acc = val_metrics['sub_acc']
        torch.save(model.state_dict(), 'best_model_2.pth')
        print("  [Saved Best Model]")
        
print("Training Complete.")

cell6: Inference & Submission

In [ ]:
# 1. Define Test Dataset Class
class MultiClassImageTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        # Only load .jpg files to avoid errors with system files
        self.image_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (64, 64))
            
        if self.transform:
            image = self.transform(image)
        return image, img_name

# 2. Setup Test Loader
# Note: Ensure 'test_images' folder exists. If you only have 'train_images' for now,
# you can change this to 'train_images' to test the code.
test_img_dir = 'test_images' 
if not os.path.exists(test_img_dir):
    print(f"Warning: {test_img_dir} not found. Using 'train_images' for demo.")
    test_img_dir = 'train_images'

test_dataset = MultiClassImageTestDataset(test_img_dir, transform=image_preprocessing)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False) # Batch size 1 for safe testing

# 3. Load Best Model
model = SoftGatedExpertModel(len(super_map_df), sub_map_df).to(device)
model.load_state_dict(torch.load('best_model_2.pth'))
model.eval() # Important: Eval mode!

# 4. Run Inference with Thresholding
# We need to determine the indices for "Novel"
novel_super_idx = 3 # 0,1,2 are known. 3 is Novel.
novel_sub_idx = len(sub_map_df) # If 0..89 are known, 90 is Novel.

# Thresholds (Tune these based on validation analysis!)

predictions = []

print(f"Running inference on {len(test_dataset)} images...")

with torch.no_grad():
    for image, img_name in test_loader:
        image = image.to(device)
        
        # Forward Pass
        gate_logits, sub_probs = model(image)
        
        # --- Super-Class Prediction ---
        gate_probs = F.softmax(gate_logits, dim=1)
        super_conf, super_pred = torch.max(gate_probs, 1)
        
        # Thresholding Logic
        if super_conf.item() < SUPER_THRESHOLD_softmax:
            final_super = novel_super_idx
            # If super is novel, sub must be novel
            final_sub = novel_sub_idx
        else:
            final_super = super_pred.item()
            
            # --- Sub-Class Prediction ---
            # We only check sub-class confidence if super-class was known
            sub_conf, sub_pred = torch.max(sub_probs, 1)
            
            if sub_conf.item() < SUB_THRESHOLD_prob:
                final_sub = novel_sub_idx
            else:
                final_sub = sub_pred.item()
        
        predictions.append({
            'image': img_name[0],
            'superclass_index': final_super,
            'subclass_index': final_sub
        })

# 5. Save Submission
submission_df = pd.DataFrame(predictions)
submission_df.to_csv('model2_submission.csv', index=False)
print("Saved 'model2_submission.csv'")
print(submission_df.head())

cell7: Visualization

In [ ]:
# Cell 7: Detailed Visualization (Probability Method)

def visualize_prob_predictions(model, test_loader, num_images=5, prob_threshold_super=SUPER_THRESHOLD_softmax, prob_threshold_sub=SUB_THRESHOLD_prob):
    model.eval()
    images_shown = 0
    
    # Create a mapping for "Novel" classes
    super_names = super_map_df['class'].tolist() + ['Novel']
    sub_names = sub_map_df['class'].tolist() + ['Novel']
    
    # Novel Indices
    novel_super_idx = 3
    novel_sub_idx = len(sub_map_df)
    
    # Create Figure with fixed size
    fig = plt.figure(figsize=(20, 6))
    
    print(f"Visualizing with Super PROB Threshold: {prob_threshold_super}")
    print(f"Visualizing with Sub  PROB Threshold: {prob_threshold_sub}")
    
    with torch.no_grad():
        for image, img_name in test_loader:
            if images_shown >= num_images:
                break
                
            image = image.to(device)
            
            # Forward Pass
            gate_logits, sub_probs = model(image)
            
            # --- 1. Super-Class Logic (Probability Based) ---
            gate_probs = F.softmax(gate_logits, dim=1)
            super_conf, super_pred_idx = torch.max(gate_probs, 1)
            super_val = super_conf.item()
            
            if super_val < prob_threshold_super:
                final_super_idx = novel_super_idx
                super_color = 'red'
            else:
                final_super_idx = super_pred_idx.item()
                super_color = 'black'
                
            # --- 2. Sub-Class Logic (Probability Based) ---
            sub_conf, sub_pred_idx = torch.max(sub_probs, 1)
            sub_val = sub_conf.item()
            
            # If Super is Novel, Sub MUST be Novel
            if final_super_idx == novel_super_idx:
                final_sub_idx = novel_sub_idx
                sub_color = 'red'
            elif sub_val < prob_threshold_sub:
                final_sub_idx = novel_sub_idx
                sub_color = 'red'
            else:
                final_sub_idx = sub_pred_idx.item()
                sub_color = 'black'

            # --- Plotting ---
            ax = plt.subplot(1, num_images, images_shown + 1)
            
            # Un-normalize image
            img_display = image.cpu().squeeze(0).permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_display = std * img_display + mean
            img_display = np.clip(img_display, 0, 1)
            
            plt.imshow(img_display)
            plt.axis('off')
            
            # Get Names
            pred_super_name = super_names[final_super_idx]
            if final_sub_idx < len(sub_names):
                pred_sub_name = sub_names[final_sub_idx]
            else:
                pred_sub_name = "Novel"
            
            # --- Text Rendering (Matching Cell 9 Format) ---
            # Super-Class
            ax.text(0.5, 1.35, 
                    f"{pred_super_name}\n(Prob: {super_val:.2f})", 
                    color=super_color, ha='center', va='bottom', 
                    transform=ax.transAxes, fontsize=11, fontweight='bold')
            
            # Sub-Class
            display_name = pred_sub_name.replace('_', '\n')
            ax.text(0.5, 1.05, 
                    f"{display_name}\n(Prob: {sub_val:.2f})", 
                    color=sub_color, ha='center', va='bottom', 
                    transform=ax.transAxes, fontsize=10)
            
            images_shown += 1

    # Wide spacing to prevent overlap
    plt.subplots_adjust(top=0.70, wspace=1.0)
    plt.show()

# Run visualization
viz_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)
visualize_prob_predictions(model, viz_loader, num_images=6)

In [ ]:
# Cell 8: Inference with Logit Energy Thresholding (Experimental)

# 1. Setup Test Loader (Same as before)
test_img_dir = 'test_images' 
if not os.path.exists(test_img_dir):
    test_img_dir = 'train_images' # Fallback for demo

test_dataset = MultiClassImageTestDataset(test_img_dir, transform=image_preprocessing)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 2. Load Best Model
model = SoftGatedExpertModel(len(super_map_df), sub_map_df).to(device)
model.load_state_dict(torch.load('best_model_2.pth'))
model.eval()

# 3. Define Thresholds
# NOTE: Logit thresholds are NOT probabilities (0-1). They can be any number (e.g., 2.5, 5.0, 10.0).
# Higher = More strict (Predicts "Novel" more often)
# Lower = Less strict (Predicts known classes more often)

novel_super_idx = 3
novel_sub_idx = len(sub_map_df)

predictions = []
novel_counts = 0

print(f"Running Logit-Based Inference on {len(test_dataset)} images...")
print(f"Super-Class Logit Threshold: {SUPER_THRESHOLD_logit}")

with torch.no_grad():
    for image, img_name in test_loader:
        image = image.to(device)
        
        # Forward Pass
        # We get gate_logits (Raw Scores) and sub_probs (Softmaxed Weighted Sums)
        gate_logits, sub_probs = model(image)
        
        # --- Super-Class Prediction (USING RAW LOGITS) ---
        # We check the magnitude of the highest logit directly
        super_logit_val, super_pred = torch.max(gate_logits, 1)
        
        if super_logit_val.item() < SUPER_THRESHOLD_logit:
            final_super = novel_super_idx
            final_sub = novel_sub_idx
            novel_counts += 1
        else:
            final_super = super_pred.item()
            
            # --- Sub-Class Prediction (Using Probs) ---
            # Note: We use probs here because our model architecture returns 'final_sub_probs'
            # (the weighted average). We don't have a single 'sub_class_logit' vector 
            # because it's a mixture of multiple experts.
            sub_conf, sub_pred = torch.max(sub_probs, 1)
            
            if sub_conf.item() < SUB_THRESHOLD_prob:
                final_sub = novel_sub_idx
            else:
                final_sub = sub_pred.item()
        
        predictions.append({
            'image': img_name[0],
            'superclass_index': final_super,
            'subclass_index': final_sub
        })

# 4. Save Submission
submission_df = pd.DataFrame(predictions)
filename = f'model2_logit_submission_thresh_{SUPER_THRESHOLD_logit}.csv'
submission_df.to_csv(filename, index=False)

print(f"Saved '{filename}'")
print(f"Total 'Novel' Super-Classes Predicted: {novel_counts}/{len(test_dataset)}")
print(submission_df.head())

In [ ]:
# Cell 9: Detailed Visualization with Color-Coded Labels (Fixed Layout - Wider Spacing)

def visualize_logit_predictions(model, test_loader, num_images=5, logit_threshold=SUPER_THRESHOLD_logit, sub_prob_threshold=SUB_THRESHOLD_prob):
    model.eval()
    images_shown = 0
    
    # Create a mapping for "Novel" classes
    super_names = super_map_df['class'].tolist() + ['Novel']
    sub_names = sub_map_df['class'].tolist() + ['Novel']
    
    # Novel Indices
    novel_super_idx = 3
    novel_sub_idx = len(sub_map_df)
    
    # Create Figure with fixed size
    # We make it wider (20) to help spacing
    fig = plt.figure(figsize=(20, 6))
    
    print(f"Visualizing with Super Logit Threshold: {logit_threshold}")
    print(f"Visualizing with Sub Prob Threshold:   {sub_prob_threshold}")
    
    with torch.no_grad():
        for image, img_name in test_loader:
            if images_shown >= num_images:
                break
                
            image = image.to(device)
            
            # Forward Pass
            gate_logits, sub_probs = model(image)
            
            # --- 1. Super-Class Logic (Logit Based) ---
            super_logit_val, super_pred_idx = torch.max(gate_logits, 1)
            super_val = super_logit_val.item()
            
            if super_val < logit_threshold:
                final_super_idx = novel_super_idx
                super_color = 'red'
            else:
                final_super_idx = super_pred_idx.item()
                super_color = 'black'
                
            # --- 2. Sub-Class Logic (Probability Based) ---
            sub_conf, sub_pred_idx = torch.max(sub_probs, 1)
            sub_val = sub_conf.item()
            
            if final_super_idx == novel_super_idx:
                final_sub_idx = novel_sub_idx
                sub_color = 'red'
            elif sub_val < sub_prob_threshold:
                final_sub_idx = novel_sub_idx
                sub_color = 'red'
            else:
                final_sub_idx = sub_pred_idx.item()
                sub_color = 'black'

            # --- Plotting ---
            ax = plt.subplot(1, num_images, images_shown + 1)
            
            # Un-normalize image
            img_display = image.cpu().squeeze(0).permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_display = std * img_display + mean
            img_display = np.clip(img_display, 0, 1)
            
            plt.imshow(img_display)
            plt.axis('off')
            
            # Get Names
            pred_super_name = super_names[final_super_idx]
            if final_sub_idx < len(sub_names):
                pred_sub_name = sub_names[final_sub_idx]
            else:
                pred_sub_name = "Novel"
            
            # --- Text Rendering ---
            # Super-Class (Bold)
            ax.text(0.5, 1.35, 
                    f"{pred_super_name}\n(Logit: {super_val:.2f})", 
                    color=super_color, ha='center', va='bottom', 
                    transform=ax.transAxes, fontsize=11, fontweight='bold')
            
            # Sub-Class (Standard)
            # We wrap the text if it's too long (simple break on underscores)
            display_name = pred_sub_name.replace('_', '\n')
            
            ax.text(0.5, 1.05, 
                    f"{display_name}\n(Prob: {sub_val:.2f})", 
                    color=sub_color, ha='center', va='bottom', 
                    transform=ax.transAxes, fontsize=10)
            
            images_shown += 1

    # FIX: Increase wspace (width space) to 1.0 or more to separate columns
    plt.subplots_adjust(top=0.70, wspace=1.0)
    plt.show()

# Run visualization
viz_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)
visualize_logit_predictions(model, viz_loader, num_images=6, 
                            logit_threshold=SUPER_THRESHOLD_logit, 
                            sub_prob_threshold=SUB_THRESHOLD_prob)

In [ ]:
# Cell 10: Compare Probability-based vs. Logit-based Predictions

# 1. Load both predictions
df_prob = pd.read_csv('model2_submission.csv')
df_logit = pd.read_csv('model2_logit_submission_thresh_2.5.csv')

# 2. Define Novel Indices (Adjust based on your map, usually 3 for Super)
NOVEL_SUPER_IDX = 3 

# 3. Calculate Statistics
prob_novel_count = len(df_prob[df_prob['superclass_index'] == NOVEL_SUPER_IDX])
logit_novel_count = len(df_logit[df_logit['superclass_index'] == NOVEL_SUPER_IDX])

print(f"--- Novelty Detection Comparison ---")
print(f"Probability Method (Cell 6): Found {prob_novel_count} novel super-classes")
print(f"Logit Energy Method (Cell 8): Found {logit_novel_count} novel super-classes")

# 4. Find Disagreements
# Where did Logit say "Novel" but Probability said "Known"?
disagreements = df_logit[
    (df_logit['superclass_index'] == NOVEL_SUPER_IDX) & 
    (df_prob['superclass_index'] != NOVEL_SUPER_IDX)
]

print(f"\nDisagreements: {len(disagreements)} images")
print("(Logit method said 'Novel', but Probability method said 'Known')")

if len(disagreements) > 0:
    print("\nSample Disagreements (Check these images manually!):")
    print(disagreements.head())
    
    # Visual check of a disagreement
    # This helps you decide which threshold method is actually smarter
    sample_img = disagreements.iloc[0]['image']
    print(f"\nChecking image: {sample_img}")
    
    # Load and display
    img_path = os.path.join(test_img_dir, sample_img)
    try:
        img = Image.open(img_path)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(f"Logit: Novel | Prob: {df_prob[df_prob['image']==sample_img]['superclass_index'].values[0]}")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Could not load image: {e}")